# Init

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import glob

import sys, os
# Repo root, found by walking up from the CWD (this notebook lives in
# notebooks/archive/), then chdir so relative data paths still resolve.
REPO = next(q for q in [Path.cwd(), *Path.cwd().parents] if (q / "idr").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)
import legacy.optimizer.optimize
import legacy.optimizer.laffont_bazin
import legacy.optimizer.optimize_sh as optimize_sh
import legacy.optimizer.optimize_inverse_rendering

# Full size sample scene

In [ ]:
albedo     = plt.imread("datasets/office/flux/albedo.webp")
relighted  = plt.imread("datasets/office/flux/relighted.webp")
relighted2 = plt.imread("datasets/office/flux/relighted2.webp")
original   = plt.imread("datasets/office/original.jpg")
original = cv.resize(original, (albedo.shape[1], albedo.shape[0]))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

imgs = [
    (original,               "original"),
    (original / albedo,      "original / albedo"),
    (relighted,              "relighted"),
    (relighted / albedo,     "relighted / albedo"),
    (relighted - original,   "relighted - original"),
]

for ax, (img, title) in zip(axes, imgs):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Down-scaled scene

In [ ]:
albedo     = plt.imread(r"datasets\office_small\albedo.webp")
relighted  = plt.imread(r"datasets\office_small\relighted.webp")
relighted2 = plt.imread(r"datasets\office_small\relighted2.webp")
original   = plt.imread(r"datasets\office_small\original.jpg")
original = cv.resize(original, (albedo.shape[1], albedo.shape[0]))

images = [original, relighted, relighted2]
labels = ["Original", "Relighted", "Relighted 2"]

## inverse rendering decomposition

In [ ]:
print("── inverse rendering decomposition (nvdiffrecmc-style) ──────────")

albedo_ir, shadings_ir, history_ir = optimize_inverse_rendering.decompose(images, normals_path="datasets/office_small/normals.png", n_iter=100,  lr=5e-2)

roughness_ir = optimize_inverse_rendering.decompose.last_result["roughness"]
metallic_ir  = optimize_inverse_rendering.decompose.last_result["metallic"]
envmaps_ir   = optimize_inverse_rendering.decompose.last_result["envmaps"]
diffuse_ir   = optimize_inverse_rendering.decompose.last_result["diffuse"]
specular_ir  = optimize_inverse_rendering.decompose.last_result["specular"]

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(28, 4))

imgs = [
    (albedo_ir,       "albedo"),
    (metallic_ir,     "metallic"),
    (roughness_ir,    "roughness"),
    (shadings_ir[2],  "shading (2)"),
    (diffuse_ir[2],   "diffuse (2)"),
    (specular_ir[2],  "specular (2)"),
    (envmaps_ir[2],   "env map (2)"),
]

for ax, (img, title) in zip(axes, imgs):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Further methods

In [ ]:
print("── Gradient-descent decomposition ──────────────────────────────")
albedo_opt, shadings_opt, history = optimize.decompose(images)


In [ ]:
print("── Laffont & Bazin ICCV 2015 ────────────────────────────────────")
albedo_lb, shadings_lb = laffont_bazin.decompose(images)


In [ ]:
print("── SH physics-based decomposition (nvdiffrecmc-style) ──────────")
albedo_sh, shadings_sh, history_sh = optimize_sh.decompose(
    images, normals_path="datasets/office/office_marigold/normals.png"
)


# Run multiple methods

In [ ]:
def show(ax, img, title):
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(title, fontsize=7)
    ax.axis("off")

N      = len(images)
imgs_f = [img.astype("float32") / 255.0 for img in images]

# shadings_lb are grayscale [H,W] → expand to [H,W,3] for display
shadings_lb_rgb = [np.stack([s]*3, axis=-1) for s in shadings_lb]

methods = [
    ("Gradient descent",          albedo_opt, shadings_opt),
    ("Laffont & Bazin 2015",      albedo_lb,  shadings_lb_rgb),
    ("SH / nvdiffrecmc-style",    albedo_sh,  shadings_sh),
]

n_cols = 1 + 2 * len(methods)   # input | (albedo + shading) × 3
fig, axes = plt.subplots(N, n_cols, figsize=(3.5 * n_cols, 3.5 * N))

for k in range(N):
    col = 0
    show(axes[k, col], imgs_f[k], f"Input\n{labels[k]}")
    col += 1
    for mname, albedo, shadings in methods:
        prefix = mname if k == 0 else ""
        show(axes[k, col],     albedo,       f"{prefix}\nAlbedo" if prefix else "")
        show(axes[k, col + 1], shadings[k],  f"{prefix}\nShading — {labels[k]}" if prefix
                                              else f"Shading — {labels[k]}")
        col += 2

plt.suptitle("Intrinsic decomposition — method comparison", y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

# ── MAE table ────────────────────────────────────────────────────────────────
hdr = f"{'':20s}" + "".join(f"  {m[0]:>26s}" for m in methods)
print(hdr)
for k in range(N):
    row = f"{labels[k]:20s}"
    for _, albedo, shadings in methods:
        mae = np.abs(albedo * shadings[k] - imgs_f[k]).mean()
        row += f"  MAE = {mae:.4f}              "
    print(row)

# ── Loss curves ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(range(0, len(history)    * 200, 200), history,    marker="o", label="Gradient descent")
axes[0].plot(range(0, len(history_sh) * 200, 200), history_sh, marker="s", label="SH / nvdiffrecmc")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Loss"); axes[0].legend()
axes[0].set_title("Optimisation loss")

# Visualise the normal map used by the SH method
normals_vis = plt.imread("marigold/normals.png")
import cv2 as cv
normals_vis = cv.resize(normals_vis, (imgs_f[0].shape[1], imgs_f[0].shape[0]))
axes[1].imshow(normals_vis.clip(0, 1))
axes[1].set_title("Marigold normal map (input to SH optimizer)")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## Pytorch3D Dataset

In [ ]:
scene_dir = os.path.dirname(image_files[0])

images       = [f for f in image_files if os.path.basename(f).startswith("rgb_")]
albedo_path  = os.path.join(scene_dir, "albedo.png")
normals_path = os.path.join(scene_dir, "normal.png")

# Multi Illu Dataset

In [ ]:
image_files = glob.glob("datasets/mit/multi_ill_dataset_small/*")

scene_dir = os.path.dirname(image_files[0])

multi_ill_images       = [f for f in image_files if os.path.basename(f).startswith("dir")]
albedo_path  = os.path.join(scene_dir, "albedo.png")
normals_path = os.path.join(scene_dir, "normal.png")

len(multi_ill_images)

In [ ]:
multi_ill_images[0]

In [ ]:
print("── SH physics-based decomposition (nvdiffrecmc-style) ──────────")
albedo_sh, shadings_sh, history_sh = optimize_sh.decompose(
    multi_ill_images, normals_path=normals_path
)


In [ ]:
# plt.imshow(shadings_sh[0])
shadings_sh[0].min(), shadings_sh[0].max()

In [ ]:
print(albedo_sh.max()) 
print(shadings_sh[0].max())

res = multi_ill_images[0] - albedo_sh * shadings_sh[0]

plt.imshow(multi_ill_images[0])
plt.show()
plt.imshow(albedo_sh * shadings_sh[0])

In [ ]:
# Load the reference normal map
normals_vis = plt.imread("multi_ill_marigold/normal.png")

# Load reference albedo and shading from marigold folder if they exist
try:
    albedo_ref = plt.imread("multi_ill_marigold/albedo.png")
    shading_ref = plt.imread("multi_ill_marigold/shading.png")
    has_ref = True
except:
    has_ref = False

# Create a comparison visualization
n_rows = 5 if not has_ref else 6
fig, axes = plt.subplots(n_rows, 2, figsize=(10, 20 if not has_ref else 24))

# Show albedo
axes[0, 0].imshow(np.clip(albedo_sh, 0, 1))
axes[0, 0].set_title("Generated Albedo")
axes[0, 0].axis("off")

# Show reference albedo if available
if has_ref:
    axes[0, 1].imshow(np.clip(albedo_ref, 0, 1))
    axes[0, 1].set_title("Reference Albedo (Marigold)")
    axes[0, 1].axis("off")
    row_offset = 1
else:
    axes[0, 1].imshow(np.clip(normals_vis, 0, 1))
    axes[0, 1].set_title("Normal Map (Marigold)")
    axes[0, 1].axis("off")
    row_offset = 1

# Show a few shadings from the decomposition
for i in range(3):
    axes[i + row_offset, 0].imshow(np.clip(shadings_sh[i], 0, 1))
    axes[i + row_offset, 0].set_title(f"Generated Shading {i}")
    axes[i + row_offset, 0].axis("off")

# Show input images on the right
for i in range(3):
    imgs_normalized = multi_ill_images[i].astype('float32') / 255.0
    axes[i + row_offset, 1].imshow(np.clip(imgs_normalized, 0, 1))
    axes[i + row_offset, 1].set_title(f"Input Image {i}")
    axes[i + row_offset, 1].axis("off")

# Show reference shading if available
if has_ref and row_offset == 1:
    axes[4, 1].imshow(np.clip(shading_ref, 0, 1))
    axes[4, 1].set_title("Reference Shading (Marigold)")
    axes[4, 1].axis("off")

plt.tight_layout()
plt.show()

# Decomposing the SH dataset

In [ ]:
image_files = glob.glob("datasets/synthetic_pytorch3d_dataset/00003/*")
image_files = [
    f for f in image_files
    if os.path.basename(f).startswith("rgb")
]
multi_ill_images = [plt.imread(f) for f in image_files]

multi_ill_images = [img[..., :3] for img in multi_ill_images] # get rid of alpha 

len(multi_ill_images)

In [ ]:
print("── SH physics-based decomposition (nvdiffrecmc-style) ──────────")
albedo_sh, shadings_sh, history_sh = optimize_sh.decompose(
    multi_ill_images[:2], normals_path=r"synthetic_pytorch3d_dataset\00003\normal.png", n_iter=200000, lr=5e-3,
              lambda_sparse=0, lambda_white=0)


In [ ]:
# no_reg = history_sh
# plt.plot(np.log(no_reg), label="no regularisation")
plt.plot(np.log(history_sh), label="SH decomposition")
plt.xlabel("iteration (×200)")
plt.ylabel("log loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# plt.imshow(multi_ill_images[2])
plt.title("predicted")
plt.imshow(albedo_sh)
plt.show()

albedo_ref = plt.imread(r"datasets/synthetic_pytorch3d_dataset\00000\albedo.png")
albedo_ref = albedo_ref[...,:3]
plt.title("original")
plt.imshow(albedo_ref)
plt.show()

plt.title("diff")
plt.imshow((albedo_ref - albedo_sh + 1) / 2)


In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np

configs = [
    {"label": "no reg",            "lambda_sparse": 0.0, "lambda_white": 0.0},
    {"label": "sparse only",       "lambda_sparse": 0.5, "lambda_white": 0.0},
    {"label": "white only",        "lambda_sparse": 0.0, "lambda_white": 0.1},
    {"label": "sparse + white",    "lambda_sparse": 0.5, "lambda_white": 0.1},
]

results = {}

for cfg in configs:
    print(f"\n── {cfg['label']} ──────────────────────────────────────────")
    albedo, shadings, history = optimize_sh.decompose(
        multi_ill_images,
        normals_path=r"datasets/synthetic_pytorch3d_dataset\00000\normal.png",
        n_iter=20000,
        lr=5e-3,
        lambda_sparse=cfg["lambda_sparse"],
        lambda_white=cfg["lambda_white"],
    )
    results[cfg["label"]] = {
        "albedo": albedo,
        "shadings": shadings,
        "history": history,
    }

# ── loss curves ───────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
for cfg in configs:
    label = cfg["label"]
    history = results[label]["history"]
    plt.plot(np.log(history), label=label)

plt.xlabel("iteration (×200)")
plt.ylabel("log loss")
plt.title("regularisation ablation")
plt.legend()
plt.tight_layout()
plt.show()

# ── recovered albedos side by side ───────────────────────────────────────────
fig, axes = plt.subplots(1, len(configs), figsize=(4 * len(configs), 4))
for ax, cfg in zip(axes, configs):
    label = cfg["label"]
    ax.imshow(results[label]["albedo"].clip(0, 1))
    ax.set_title(label)
    ax.axis("off")
plt.suptitle("recovered albedo")
plt.tight_layout()
plt.show()

# Generate & Recover torch3d dataset

In [ ]:
import glob, os
import numpy as np
import matplotlib.pyplot as plt
from legacy.optimizer import optimize_sh

# ── config ────────────────────────────────────────────────────────────────────
DATASET_DIR   = "datasets/synthetic_pytorch3d_dataset"
N_ITER        = 20000
LR            = 5e-3
IMAGE_COUNTS  = [2, 3, 10, 20]   # 20 = "all"

reg_configs = [
    {"label": "no reg",         "lambda_sparse": 0.0, "lambda_white": 0.0},
    {"label": "sparse only",    "lambda_sparse": 0.5, "lambda_white": 0.0},
    {"label": "white only",     "lambda_sparse": 0.0, "lambda_white": 0.1},
    {"label": "sparse + white", "lambda_sparse": 0.5, "lambda_white": 0.1},
]

# ── discover scenes ───────────────────────────────────────────────────────────
scene_dirs = sorted([
    d for d in glob.glob(os.path.join(DATASET_DIR, "*"))
    if os.path.isdir(d)
])
print(f"found {len(scene_dirs)} scenes")

# ── main loop ─────────────────────────────────────────────────────────────────
# results[scene_id][n_imgs][reg_label] = {"albedo", "history"}
all_results = {}

for scene_dir in scene_dirs:
    scene_id = os.path.basename(scene_dir)
    print(f"\n{'═'*60}")
    print(f"  scene {scene_id}")
    print(f"{'═'*60}")

    normals_path = os.path.join(scene_dir, "normal.png")

    # load all rgb images for this scene
    rgb_files = sorted([
        f for f in glob.glob(os.path.join(scene_dir, "*.png"))
        if os.path.basename(f).startswith("rgb")
    ])
    all_images = [plt.imread(f)[..., :3] for f in rgb_files]
    print(f"  {len(all_images)} rgb images available")

    all_results[scene_id] = {}

    for n_imgs in IMAGE_COUNTS:
        imgs = all_images[:n_imgs]
        print(f"\n  ── n_imgs={n_imgs} ───────────────────────────────────")
        all_results[scene_id][n_imgs] = {}

        for cfg in reg_configs:
            print(f"     {cfg['label']}")
            albedo, shadings, history = optimize_sh.decompose(
                imgs,
                normals_path=normals_path,
                n_iter=N_ITER,
                lr=LR,
                lambda_sparse=cfg["lambda_sparse"],
                lambda_white=cfg["lambda_white"],
            )
            all_results[scene_id][n_imgs][cfg["label"]] = {
                "albedo":  albedo,
                "history": history,
            }

# ── plot 1: loss curves — one figure per scene, rows=n_imgs, cols=reg ─────────
for scene_id, scene_res in all_results.items():
    fig, axes = plt.subplots(
        len(IMAGE_COUNTS), len(reg_configs),
        figsize=(4 * len(reg_configs), 3 * len(IMAGE_COUNTS)),
        sharey="row",
    )
    fig.suptitle(f"loss curves — scene {scene_id}", fontsize=13)

    for r, n_imgs in enumerate(IMAGE_COUNTS):
        for c, cfg in enumerate(reg_configs):
            ax = axes[r, c]
            history = scene_res[n_imgs][cfg["label"]]["history"]
            ax.plot(np.log(history))
            if r == 0:
                ax.set_title(cfg["label"], fontsize=9)
            if c == 0:
                ax.set_ylabel(f"n={n_imgs}\nlog loss", fontsize=8)
            ax.set_xlabel("iter ×200", fontsize=7)

    plt.tight_layout()
    plt.savefig(f"loss_curves_{scene_id}.png", dpi=100)
    plt.show()

# ── plot 2: recovered albedos — one figure per scene, rows=n_imgs, cols=reg ──
for scene_id, scene_res in all_results.items():
    fig, axes = plt.subplots(
        len(IMAGE_COUNTS), len(reg_configs),
        figsize=(4 * len(reg_configs), 4 * len(IMAGE_COUNTS)),
    )
    fig.suptitle(f"recovered albedo — scene {scene_id}", fontsize=13)

    for r, n_imgs in enumerate(IMAGE_COUNTS):
        for c, cfg in enumerate(reg_configs):
            ax = axes[r, c]
            albedo = scene_res[n_imgs][cfg["label"]]["albedo"]
            ax.imshow(albedo.clip(0, 1))
            ax.axis("off")
            if r == 0:
                ax.set_title(cfg["label"], fontsize=9)
            if c == 0:
                ax.set_ylabel(f"n={n_imgs}", fontsize=9)

    plt.tight_layout()
    plt.savefig(f"albedo_{scene_id}.png", dpi=100)
    plt.show()

# ── plot 3: final loss summary — heatmap per scene ────────────────────────────
fig, axes = plt.subplots(
    1, len(scene_dirs),
    figsize=(3 * len(scene_dirs), 3 * len(IMAGE_COUNTS) / 2),
)
if len(scene_dirs) == 1:
    axes = [axes]

for ax, scene_dir in zip(axes, scene_dirs):
    scene_id = os.path.basename(scene_dir)
    matrix = np.zeros((len(IMAGE_COUNTS), len(reg_configs)))
    for r, n_imgs in enumerate(IMAGE_COUNTS):
        for c, cfg in enumerate(reg_configs):
            history = all_results[scene_id][n_imgs][cfg["label"]]["history"]
            matrix[r, c] = history[-1]

    im = ax.imshow(matrix, aspect="auto")
    ax.set_xticks(range(len(reg_configs)))
    ax.set_xticklabels([c["label"] for c in reg_configs], rotation=30, ha="right", fontsize=7)
    ax.set_yticks(range(len(IMAGE_COUNTS)))
    ax.set_yticklabels([f"n={n}" for n in IMAGE_COUNTS])
    ax.set_title(f"scene {scene_id}\nfinal loss", fontsize=9)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig("final_loss_heatmap.png", dpi=100)
plt.show()

In [ ]:
import pickle

with open("all_results.pkl", "wb") as f:
    pickle.dump(all_results, f)


In [ ]:
with open("all_results.pkl", "rb") as f:
    test = pickle.load(f)

# Synthetic dataset forward rendering experiments

In [ ]:
normals = plt.imread("datasets/office_small/normals.png")
# albedo = plt.imread("datasets/office_small/albedo.webp")
albedo = np.ones_like(normals)
# albedo = np.random.random(normals.shape)


In [ ]:
import torch
from optimize_inverse_rendering import render_environment_pbr

torch.manual_seed(42)

total, diffuse, specular = render_environment_pbr(
    normals= torch.from_numpy(normals),
    base_color=torch.from_numpy(albedo),
    roughness=torch.zeros_like(torch.from_numpy(albedo)[...,0]),
    metallic=torch.ones_like(torch.from_numpy(albedo)[...,0])*0.5,    
    # envmap=torch.full((12,24,3),0.5),
    envmap=torch.rand(12,24,3),
    view_dir=torch.tensor([0,0,1]))

plt.imshow(total)

In [ ]:
import numpy as np

scale=np.array([1.5195003, 1.9376863, 1.6935806])

sh_coeffs_est = np.load(r"results/archive/raw_optimizer\scene_0000\no reg\sh_coeffs_est.npy")
sh_coeffs_gt = np.load(r"results/archive/raw_optimizer\scene_0000\no reg\sh_coeffs_gt.npy")


sh_coeffs_est / scale